# 04 — Build benchmark

Assemble reviewed items into an immutable versioned release at `benchmark/vN.N/`.

**Hard-fails** on any item carrying `llm_draft` or a missing `label_source`, and refuses to write a prefix that already exists. A correction is a new version, never an edit.

Read the `labeling-controls` skill. `/cut-version` runs this gate.

In [ ]:
# Colab bootstrap. Run once per runtime.
!pip install -q google-cloud-storage jsonschema

from google.colab import auth
auth.authenticate_user()

import sys, pathlib
GITHUB_USER = 'ajrngn'
REPO = pathlib.Path('/content/auditagent-bench')
if not REPO.exists():
    !git clone -q https://github.com/{GITHUB_USER}/auditagent-bench.git {REPO}
sys.path.insert(0, str(REPO))

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
BUCKET = 'audit-agent-500113-data'
PROCESSED_PREFIX = 'processed'
BENCHMARK_PREFIX = 'benchmark'
DATASET_VERSION = 'v0.1'

In [ ]:
from src import gcs, schema

TARGET = f'{BENCHMARK_PREFIX}/{DATASET_VERSION}'

# Refuse to overwrite a released version. Raises if the prefix is occupied.
gcs.require_absent(BUCKET, TARGET)

items = list(gcs.read_jsonl(BUCKET, f'{PROCESSED_PREFIX}/labels/reviewed.jsonl'))
print(f'{len(items)} candidate items for {TARGET}')

In [ ]:
passed, report = schema.release_gate(items)
print('\n'.join(report))
print()
print('GATE PASSED' if passed else 'GATE FAILED — nothing will be written')

In [ ]:
# Write only on a full pass. A failing label goes back to human review; it is
# never relabeled to make the gate green.
if not passed:
    raise SystemExit('release gate failed')

n = gcs.write_jsonl(BUCKET, f'{TARGET}/items.jsonl', items)
gcs.write_json(BUCKET, f'{TARGET}/_manifest.json', {
    'version': DATASET_VERSION,
    'n_items': n,
    'gate_report': report,
})
print(f'wrote {n} items to {TARGET}')
print('Now append an entry to docs/versions.md.')